Un **sistema software auto-adattivo** è capace di modificare il proprio funzionamento **runtime** in risposta a cambiamenti:
- nel proprio **stato interno**
- **nell'ambiente esterno** in cui opera

L'obiettivo di questi sistemi è continuare a fornire il servizio e rispettare determinati requisiti, richiedendo un intervento umano limitato (es. un'applicazione cloud può aumentare il numero di istanze in risposta a un aumento del carico di lavoro).

Si collegano questi sistemi al concetto più generale di **autonomic computing**, che a sua volta è un paradigma ispirato al sistema nervoso umano per cui alcune funzioni corporee (battito cardiaco, respirazione, digestione) sono regolate automaticamente senza intervento cosciente.  
Analogamente un sistema informatico autonomo gestisce da solo alcuni aspetti del proprio funzionamento. 

Chiaramente gli **obiettivi e le regole devono essere definite dai progettisti o dagli amministratori del sistema**.  
I principi di auto-adattamento si applicano in molti contesti, a noi interessano in particolare applicati al cloud, all'edge computing e al computing continuum.

Alcuni esempi di sistemi software auto-adattivi:

<img src="imgs/auto.png" width="400">

es. Kubertenes è un sistema di orchestrazione che gestisce applicazioni eseguite in container su un insieme di nodi. Tra le attività di adattamento si hanno ad es. placement o scheduling (decidere su quale nodo eseguire un container), scaling orizzontale (aumentare o diminuire il numero di istanze di un'applicazione), verticale (aumentare o diminuire le risorse per un singolo componente) e healing (ricreare su nodi disponibili componenti che erano eseguiti in un nodo ora guasto).

es. Amazon EC2 Auto Scaling è un servizio che permette di aumentare o diminuire automaticamente il numero di istanze EC2 in base a regole definite dall'utente, come ad esempio l'utilizzo della CPU o il numero di richieste in coda. **Il sistema adatta automaticamente le risorse per mantenere il comportamento desiderato**.

Si citano anche tra gli esempi reti che si riconfigurano per reagire ai guasti (l'abbiamo visto in dettaglio con Chord) e database capaci di gestire automaticamente le risorse.

## self-* capabilities

Esistono diverse capacità di autogestione, raggruppate nella notazione **self-{*}**:
1. **Self-Optimization**: ottimizzare prestazioni o uso delle risorse, rispettando gli obiettivi di qualità (es. cambiare il posizionamento dei componenti per ridurre i tempi di risposta);
2. **Self-Healing**: rilevare, diagnosticare e recuperare da guasti (es. escludere un nodo guasto dal bilanciamento del carico);
3. **Self-Configuration**: integrare nuovi elementi o modificare automaticamente parametri di configurazione (es. aggiungere un nuovo nodo al servizio);
4. **Self-Protection**: rilevare minacce e reagire per limitare gli effetti (es. identificare un attacco DDoS e bloccare il traffico malevolo).

Per raggiungere questi obiettivi, il sistema deve:
- **conoscere il proprio stato interno (self-awareness)** quindi risorse disponibili, prestazioni, componenti attivi etc..
- **conoscere il contesto operativo (self-situation)** quindi condizioni della rete, utenti, carico esterno e ambiente
- **rilevare i cambiamenti (self-monitoring)**, osservare cioè come evolvono sistema e contesto
- **intervenire (self-adjustement)**, modificando il proprio comportamento in risposta a ciò che è stato rilevato.

Concettualmente il modello di un sistema self-adaptive è fatto di due parti:
- **Managed system**: è il software che fornisce il servizio e che deve essere adattato (es. un'applicazione Web con più istanze)
- **Managing system**: è il sistema di gestione vero e proprio, che osserva il managed system e decide come adattarlo per perseguire gli obiettivi prestabiliti.

Tra i due sistemi, serve una strumentazione che permetta di **misurare** (attraverso sensori o sonde) e **intervenire** nel caso in cui sia necessario adattare il sistema attraverso meccanismi di intervento specifici (**effettori**).

Il managing system, oltre che il managed system, deve osservare anche l'ambiente in il managed system si trova (quindi ciò che non è sotto il suo controllo diretto come utenti, rete, software esterno etc..)

<img src="imgs/self-adaptive.png" width="600">

es. in un'applicazione cloud il sistema gestito serve le richieste; le sonde misurano il carico, la CPU e i tempi di risposta; il sistema di gestione valuta se le risorse sono sufficienti gli effettori possono creare nuove repliche

Si tratta di un così detto **ciclo di controllo a retroazione (feedback loop)**. 

## MAPE-K
Questo ciclo è organizzato nel modello MAPE:
$$ \text{Monitor} \rightarrow \text{Analyze} \rightarrow \text{Plan} \rightarrow \text{Execute}$$
Queste quattro fasi usano una base di conoscenza, **Knowledge**, da cui la denominazione MAPE-K.

In generale:
- **Monitor** raccoglie i dati dal sistema e dall'ambiente. Può filtrarli, aggregarli e correrarli per ottenere informazioni più utili da analizzare;
- **Analyze** valuta le informazioni raccolte e determina se è necessario un adattamento. Può usare modelli predittivi, regole o tecniche di machine learning;
- **Plan** è il cuore decisionale del ciclo: si sceglie quale azione effettuare per ottenere il cambiamento desiderato.
- **Execute** applica le azioni pianificate al sistema gestito, usando gli **effettori**.
- **Knowledge**: conserva la conoscenza condivisa necessaria per le varie fasi (es. dati di monitoraggio, stato e caratteristiche del sistema, informazioni sull'ambiente e obiettivi dell'amministratore. Non deve essere per forza un unico database centrale, è una componente logica).

es. tempi di risposta troppo alti --> il monitor raccoglie i tempi di risposta e l'utilizzo delle risorse, l'analyze si accorge che il servizio potrebbe non rispettare gli obiettivi di qualità, il plan decide quante repliche aggiungere e l'execute le avvia.

### MAPE: Monitor
Per progettare il monitoraggio ci si devono porre alcune domande:
- **Quando (When) Monitorare?** Il monitoraggio può infatti essere **continuativo** permettendo l'osservazione nel tempo tramite campionamenti regolari, oppure **on-demand** (attivato se richiesto, magari con un evento specifico);
- **Cosa (What) Monitorare?** Si possono monitorare diverse misure:
    - risorse (CPU, memoria..)
    - carico di lavoro (numero di richieste, dimensione dei dati..)
    - prestazioni (tempi di risposta, throughput..)
    - disponibilità e stato dei componenti
- **Come (How) organizzare il monitoraggio?** In un'architettura **centralizzata**, i dati confluiscono in un'entità centrale.  
In una **decentralizzata** la raccolta e l'elaborazione sono distribuite tra più componenti.  
La scelta del **come monitorare** invece è distinta dalla metodologia di misurazione, che può essere:
    - **Passiva**: si osservano i dati o attività già disponibili senza iniettare nuovo traffico nel sistema 
    - **Attiva**: si inviano richieste di misurazione (es. ping, heartbeat, test di carico) per ottenere informazioni più dettagliate (carico aggiuntivo maggiore rispetto al passivo!)
- **Dove (Where) conservare i dati e come prepararli?** es. database a serie temporali come InfluxDB, adatti a misure associate nel tempo come utilizzo della CPU che viene rilevato ogni pochi secondi...  
Sul come prepararli, come anticipato, i dati possono essere filtrati, aggregati, correlati o normalizzati

### MAPE: Analyze
L'analisi come detto usa i dati monitorati per stabilire se serve un adattamento.

La prima scelta riguarda quando attivare l'analisi:
- **a tempo** (periodicamente, es. ogni 2 minuti)
- **a evento** (quando un evento rilevante si verifica, es. un certo numero di richieste in coda)

La seconda distinzione è tra approccio reattivo e proattivo:
- **reattivo**: si reagisce a un cambiamento già avvenuto (es. il carico è aumentato e il sistema rileva che servono più risorse)
- **proattivo**: si prevede la necessità futura di adattamento e si prepara l'intervento in anticipo. Il motivo è che **creare risorse richiede tempo**, e un intervento proattivo può compensare questo ritardo. La previsione può usare tecniche statistiche o di ML (da modelli semplici tipo regressione lineare a reti neurali complesse). Essendo una previsione ovviamente è soggetta a possibili errori.